# Variational Autoencoders

VAEs (Variational Autoencoders) were designed to solve the problem of latent space by turning it into a continuous, searchable landscape.

### The core difference: Discrete vs. Probabilistic

The fundamental shift from a standard Autoencoder to a VAE is how they represent the "bottleneck".

1. Standard Autoencoder (Discrete)
- It maps an input to a single fixed point in the latent space. The space between these points is undefined, which is the main problem.

2. Variational Autoencoder (Probabilistic)
- Instead of mapping input to one point, the encoder maps input to a probability distribution (usually a Normal/Gaussian distribution).

### Importance of VAEs
The need for VAEs arises when you want to do more than just reconstruct; you want to **generate**.
- **Generative Modeling**: An entirely new sample/data that looks like the training set but has never existed before, can be created through VAEs
- **Interpolation**: We can walk though the latent space. (e.g. If you have a point for a "smiling face" and a point for a "neutral face," you can find the points in between to generate a face that is slowly starting to smile.)

- **Disentanglement**: VAEs can be tuned so that specific dimensions in the latent space correspond to specific features (e.g. one number controls "rotation", another controls "color").

### Reparameterization trick

In [6]:
import torch
import numpy as np
import torch.nn as nn

In [2]:
def forward(self, x):
    base_out = self.base_model(x)
        
    self.mu = self.lin_mu(base_out)
    self.log_var = self.lin_var(base_out)
    std = torch.exp(self.log_var/2)
                
    eps = torch.randn_like(self.mu)
    z = self.mu + eps * std
    return z

In [3]:
def sample(self, sample_shape=torch.Size()):
    shape = self._extended_shape(sample_shape)
    with torch.no_grad():
        return torch.normal(self.loc.expand(shape),
                            self.scale.expand(shape))

def rsample(self, sample_shape=torch.Size()):
    shape = self._extended_shape(sample_shape)
    eps = _standard_normal(shape,
                           dtype=self.loc.dtype,
                           device=self.loc.device)
    return self.loc + eps * self.scale

### KL Divergence loss

In [5]:
def kl_div(mu, std):
    kl_div = -0.5*(1+np.log(std**2) - mu**2 - std**2)
    return kl_div

In [ ]:
class EncoderVar(nn.Module):
    def __init__(self, input_shape, z_size, base_model):
        super().__init__()
        self.z_size = z_size
        self.input_shape = input_shape
        self.base_model = base_model
        output_size = self.get_output_size()
        self.lin_mu = nn.Linear(output_size, z_size)
        self.lin_var = nn.Linear(output_size, z_size)
    
    def get_output_size(self):
        device = next(self.base_model.parameters()).device.type
        size = self.base_model(torch.zeros(1, *self.input_shape, device=device)).size(1)
        return size

    def kl_loss(self):
        kl_loss = -0.5*(1 + self.log_var - self.mu**2 - torch.exp(self.log_var))
        return kl_loss

    def forward(self, x):
        base_out = self.base_model(x)
        self.mu = self.lin_mu(base_out)
        self.log_var = self.lin_var(base_out)
        std = torch.exp(self.log_var/2)

        eps = torch.rand_like(self.mu)
        z = self.mu + eps * std
        return z

In [8]:
def set_seed(self, seed=42):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(13)
input_shape = (1, 28, 28)
z_size = 1

base_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(np.prod(input_shape), 2048),
    nn.LeakyReLU(),
    nn.Linear(2048, 2048),
    nn.LeakyReLU(),
)

encoder_var = EncoderVar(input_shape, z_size, base_model)

decoder_var = nn.Sequential(
    # z_size -> 2048
    nn.Linear(z_size, 2048),
    nn.LeakyReLU(),
    # 2048 -> 2048
    nn.Linear(2048, 2048),
    nn.LeakyReLU(),
    # 2048 -> C*H*W
    nn.Linear(2048, np.prod(input_shape)),
    # C*H*W -> (C, H, W)
    nn.Unflatten(1, input_shape)
)

class AutoEncoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.enc = encoder
        self.dec = decoder
        
    def forward(self, x):
        # when encoder met decoder
        enc_out = self.enc(x)
        return self.dec(enc_out)
    

model_vae = AutoEncoder(encoder_var, decoder_var)

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import TensorDataset, DataLoader

def draw_circle(radius, center_x=0.5, center_y=0.5, size=28):
    # draw a circle using coordinates for the center, and the radius
    circle = plt.Circle((center_x, center_y), radius, color='k', fill=False)
    fig, ax = plt.subplots(figsize=(1, 1))
    ax.add_patch(circle)
    ax.axis('off')
    buf = fig.canvas.print_to_buffer()
    plt.close()
    # converts matplotlib figure into PIL image, make it grayscale, and resize it
    return np.array(Image.frombuffer('RGBA', buf[1], buf[0]).convert('L').resize((int(size), int(size))))

def gen_circles(n, size=28):
    # generates random coordinates around (0.5, 0.5) as center points
    center_x = np.random.uniform(0.0, 0.03, size=n).reshape(-1, 1)+.5
    center_y = np.random.uniform(0.0, 0.03, size=n).reshape(-1, 1)+.5
    # generates random radius sizes between 0.03 and 0.47
    radius = np.random.uniform(0.03, 0.47, size=n).reshape(-1, 1)
    sizes = np.ones((n, 1))*size

    coords = np.concatenate([radius, center_x, center_y, sizes], axis=1)
    # generates circles using draw_circle function
    circles = np.apply_along_axis(func1d=lambda v: draw_circle(*v), axis=1, arr=coords)
    return circles, radius

np.random.seed(42)
# generates 1,000 circles
circles, radius = gen_circles(1000)

circles_ds = TensorDataset(torch.as_tensor(circles).unsqueeze(1).float()/255, torch.as_tensor(radius))
circles_dl = DataLoader(circles_ds, batch_size=32, shuffle=True, drop_last=True)

In [10]:
x, y = next(iter(circles_dl))
zs = encoder_var(x)
reconstructed = decoder_var(zs)

In [11]:
loss_fn_raw = nn.MSELoss(reduction='none')
raw_mse = loss_fn_raw(reconstructed, x)
raw_mse.shape

torch.Size([32, 1, 28, 28])

In [12]:
raw_mse.sum(), nn.MSELoss(reduction='sum')(reconstructed, x)

(tensor(24247.6309, grad_fn=<SumBackward0>),
 tensor(24247.6309, grad_fn=<MseLossBackward0>))

In [13]:
sum_over_pixels = raw_mse.sum(dim=[1, 2, 3])
sum_over_pixels.mean()

tensor(757.7385, grad_fn=<MeanBackward0>)

In [14]:
raw_kl = encoder_var.kl_loss()
raw_kl.shape

torch.Size([32, 1])

In [15]:
for raw, d in zip([raw_mse, raw_kl], [[1, 2, 3], 1]):
    print(f'{raw.mean(dim=d).mean(dim=0).item():.4f}, {raw.mean(dim=d).sum(dim=0).item():.4f}, '
          f'{raw.sum(dim=d).mean(dim=0).item():.4f}, {raw.sum(dim=d).sum(dim=0).item():.4f}')

0.9665, 30.9281, 757.7385, 24247.6328
0.0062, 0.1998, 0.0062, 0.1998


In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_vae.to(device)
loss_fn = nn.MSELoss(reduction='none')
optim = torch.optim.Adam(model_vae.parameters(), 0.0003)

num_epochs = 30

train_losses = []

reconstruction_loss_factor = 1

for epoch in range(1, num_epochs+1):
    batch_losses = []
    for i, (x, _) in enumerate(circles_dl):
        model_vae.train()
        x = x.to(device)

        # Step 1 - Computes our model's predicted output - forward pass
        yhat = model_vae(x)

        # Step 2 - Computes the loss
        # reduce (sum) over pixels (dim=[1, 2, 3])
        # and then reduce (sum) over batch (dim=0)
        loss = loss_fn(yhat, x).sum(dim=[1, 2, 3]).sum(dim=0)
        # reduce (sum) over z (dim=1)
        # and then reduce (sum) over batch (dim=0)
        kl_loss = model_vae.enc.kl_loss().sum(dim=1).sum(dim=0)
        # we're adding the KL loss to the original MSE loss
        total_loss = reconstruction_loss_factor * loss + kl_loss

        # Step 3 - Computes gradients
        total_loss.backward()
        # Step 4 - Updates parameters using gradients and the learning rate
        optim.step()
        optim.zero_grad()
        
        batch_losses.append(np.array([total_loss.data.item(), loss.data.item(), kl_loss.data.item()]))

    # Average over batches
    train_losses.append(np.array(batch_losses).mean(axis=0))

    print(f'Epoch {epoch:03d} | Loss >> {train_losses[-1][0]:.4f}/{train_losses[-1][1]:.4f}/{train_losses[-1][2]:.4f}')

Epoch 001 | Loss >> 2048.9169/2003.9842/44.9327
Epoch 002 | Loss >> 140.2264/134.1590/6.0674
Epoch 003 | Loss >> 126.6059/125.9904/0.6155
Epoch 004 | Loss >> 123.6817/123.5897/0.0920
Epoch 005 | Loss >> 124.2495/124.1663/0.0833
Epoch 006 | Loss >> 123.9132/123.8041/0.1092
Epoch 007 | Loss >> 124.6427/124.5735/0.0692
Epoch 008 | Loss >> 128.7268/128.5154/0.2115
Epoch 009 | Loss >> 127.7835/127.5871/0.1964
Epoch 010 | Loss >> 127.3568/127.0787/0.2781
Epoch 011 | Loss >> 126.9806/126.8075/0.1732
Epoch 012 | Loss >> 129.1535/128.8505/0.3030
Epoch 013 | Loss >> 123.7204/123.6090/0.1114
Epoch 014 | Loss >> 125.3969/125.2771/0.1198
Epoch 015 | Loss >> 124.0710/123.9441/0.1269
Epoch 016 | Loss >> 126.5937/126.4686/0.1251
Epoch 017 | Loss >> 128.6493/128.2512/0.3981
Epoch 018 | Loss >> 128.1584/127.4918/0.6665
Epoch 019 | Loss >> 127.9306/127.6755/0.2551
Epoch 020 | Loss >> 125.2906/125.1844/0.1062
Epoch 021 | Loss >> 124.5838/124.4732/0.1107
Epoch 022 | Loss >> 126.4508/126.3271/0.1236
Epoch 0

### Convolutional Variational AutoEncoder (CVAE)

In [17]:
set_seed(13)

z_size = 1
n_filters = 32
in_channels = 1
img_size = 28
input_shape = (in_channels, img_size, img_size)

base_model = nn.Sequential(
    # in_channels@28x28 -> n_filters@28x28
    nn.Conv2d(in_channels, n_filters, kernel_size=3, stride=1, padding=1),
    nn.LeakyReLU(),
            
    # n_filters@28x28 -> (n_filters*2)@14x14
    nn.Conv2d(n_filters, n_filters*2, kernel_size=3, stride=2, padding=1),
    nn.LeakyReLU(),
    
    # (n_filters*2)@14x14 -> (n_filters*2)@7x7
    nn.Conv2d(n_filters*2, n_filters*2, kernel_size=3, stride=2, padding=1),
    nn.LeakyReLU(),
    
    # (n_filters*2)@7x7 -> (n_filters*2)@7x7
    nn.Conv2d(n_filters*2, n_filters*2, kernel_size=3, stride=1, padding=1),
    nn.LeakyReLU(),
    
    # (n_filters*2)@7x7 -> (n_filters*2)*7*7
    nn.Flatten(),
)

encoder_var_cnn = EncoderVar(input_shape, z_size, base_model)

In [18]:
encoder_var_cnn

EncoderVar(
  (base_model): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): LeakyReLU(negative_slope=0.01)
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): LeakyReLU(negative_slope=0.01)
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): LeakyReLU(negative_slope=0.01)
    (8): Flatten(start_dim=1, end_dim=-1)
  )
  (lin_mu): Linear(in_features=3136, out_features=1, bias=True)
  (lin_var): Linear(in_features=3136, out_features=1, bias=True)
)

In [19]:
decoder_cnn = nn.Sequential(
    # z_size -> (n_filters*2)*7*7
    nn.Linear(z_size, (n_filters*2)*int(img_size/4)**2),
    
    # (n_filters*2)*7*7 -> (n_filters*2)@7x7
    nn.Unflatten(1, (n_filters*2, int(img_size/4), int(img_size/4))),
    
    # (n_filters*2)@7x7 -> (n_filters*2)@7x7
    nn.ConvTranspose2d(n_filters*2, n_filters*2, kernel_size=3, stride=1, padding=1, output_padding=0),
    nn.LeakyReLU(),

    # (n_filters*2)@7x7 -> (n_filters*2)@14x14
    nn.ConvTranspose2d(n_filters*2, n_filters*2, kernel_size=3, stride=2, padding=1, output_padding=1),
    nn.LeakyReLU(),
    
    # (n_filters*2)@15x15 -> n_filters@28x28
    nn.ConvTranspose2d(n_filters*2, n_filters, kernel_size=3, stride=2, padding=1, output_padding=1),
    nn.LeakyReLU(),
    
    # n_filters@28x28 -> in_channels@28x28
    nn.ConvTranspose2d(n_filters, in_channels, kernel_size=3, stride=1, padding=1, output_padding=0),
    nn.Sigmoid(),
)

### Model training

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_vae_cnn = AutoEncoder(encoder_var_cnn, decoder_cnn)
model_vae_cnn.to(device)
loss_fn = nn.MSELoss(reduction='none')
optim = torch.optim.Adam(model_vae_cnn.parameters(), 0.0003)

num_epochs = 30

train_losses = []

reconstruction_loss_factor = 1

for epoch in range(1, num_epochs+1):
    batch_losses = []
    for i, (x, _) in enumerate(circles_dl):
        model_vae_cnn.train()
        x = x.to(device)

        # Step 1 - Computes our model's predicted output - forward pass
        yhat = model_vae_cnn(x)

        # Step 2 - Computes the loss
        loss = loss_fn(yhat, x).sum(dim=[1, 2, 3]).sum(dim=0)
        kl_loss = model_vae_cnn.enc.kl_loss().sum(dim=1).sum(dim=0)
        total_loss = reconstruction_loss_factor * loss + kl_loss

        # Step 3 - Computes gradients
        total_loss.backward()

        # Step 4 - Updates parameters using gradients and the learning rate
        optim.step()
        optim.zero_grad()
        
        batch_losses.append(np.array([total_loss.data.item(), loss.data.item(), kl_loss.data.item()]))

    # Average over batches
    train_losses.append(np.array(batch_losses).mean(axis=0))

    print(f'Epoch {epoch:03d} | Loss >> {train_losses[-1][0]:.4f}/{train_losses[-1][1]:.4f}/{train_losses[-1][2]:.4f}')

Epoch 001 | Loss >> 1923.2826/1843.7633/79.5193
Epoch 002 | Loss >> 168.7242/155.7002/13.0240
Epoch 003 | Loss >> 142.1335/139.3703/2.7632
Epoch 004 | Loss >> 135.2419/134.1590/1.0829
Epoch 005 | Loss >> 130.7497/129.9924/0.7573
Epoch 006 | Loss >> 127.3442/127.0603/0.2838
Epoch 007 | Loss >> 125.9245/125.7134/0.2111
Epoch 008 | Loss >> 124.1543/124.0482/0.1062
Epoch 009 | Loss >> 123.5359/123.4929/0.0431
Epoch 010 | Loss >> 122.6834/122.6430/0.0404
Epoch 011 | Loss >> 122.0418/122.0174/0.0244
Epoch 012 | Loss >> 121.0780/121.0410/0.0370
Epoch 013 | Loss >> 119.2761/118.7955/0.4806
Epoch 014 | Loss >> 113.3301/110.3172/3.0130
Epoch 015 | Loss >> 107.6557/103.4702/4.1855
Epoch 016 | Loss >> 103.7221/99.1172/4.6050
Epoch 017 | Loss >> 101.1547/95.5899/5.5648
Epoch 018 | Loss >> 98.1543/91.4428/6.7115
Epoch 019 | Loss >> 93.7019/86.0707/7.6312
Epoch 020 | Loss >> 91.3617/82.4453/8.9164
Epoch 021 | Loss >> 87.8739/78.0995/9.7745
Epoch 022 | Loss >> 84.8244/74.2214/10.6031
Epoch 023 | Loss 